## 手写BERT


In [13]:
from IPython.lib.pretty import MAX_SEQ_LENGTH
from torch import nn


class Encoder(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, d_ff, n_heads) for _ in range(n_encoder_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(q=x))
        x = self.norm2(x + self.ffn(x))
        return x


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dim: int, output_dim: int, num_heads: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.attn_dim = attn_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = attn_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)

        self.out_proj = nn.Linear(self.attn_dim, self.output_dim, bias=False)

    def forward(self, q, k=None, v=None, mask=None):
        if k is None: k = q
        if v is None: v = q

        batch_size, seq_len_q, _ = q.shape
        seq_len_k = k.shape[1]

        q = self.q_proj(q).view(batch_size, seq_len_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(k).view(batch_size, seq_len_k, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(v).view(batch_size, v.shape[1], self.num_heads, self.head_dim).transpose(1, 2)

        attn_score = torch.matmul(q, k.transpose(-2, -1))
        attn_score = attn_score / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))

        if mask is not None:
            mask = mask.unsqueeze(0).unsqueeze(1)
            attn_score = attn_score.masked_fill(mask == 0, -1e9)

        attn_weight = torch.softmax(attn_score, dim=-1)

        output = torch.matmul(attn_weight, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.attn_dim)
        return self.out_proj(output)



In [12]:
d_model = 64
d_ff = d_model * 2
n_heads = 2
n_encoder_layers = 2


class BertModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.segment_embedding = nn.Embedding(2, d_model)
        self.encoder = Encoder(d_model, d_ff, n_heads, n_encoder_layers)
        self.mlm_head = nn.Sequential(
            nn.Linear(d_model, vocab_size)
        )
        self.nsp_head = nn.Sequential(
            nn.Linear(d_model, 2)
        )

    def forward(self, input_ids, segment_ids, attention_mask):
        batch_size, seq_len = input_ids.shape

        token_embedding = self.token_embedding(input_ids)

        pos_ids = torch.arange(0, seq_len).unsqueeze(0).repeat(batch_size, 1)
        pos_embeddings = self.position_embedding(pos_ids)

        segment_embedding = self.segment_embedding(segment_ids)
        embeddings = token_embedding + pos_embeddings + segment_embedding

        decoder_output = self.encoder(embeddings)
        cls_output = decoder_output[:, 0, :]

        mlm_output = self.mlm_head(decoder_output)
        nsp_output = self.nsp_head(cls_output)

        return mlm_output, nsp_output

In [14]:
text = """
臣密言：臣以险衅，夙遭闵凶。生孩六月，慈父见背；行年四岁，舅夺母志。
祖母刘愍臣孤弱，躬亲抚养。
臣少多疾病，九岁不行，零丁孤苦，至于成立。
既无伯叔，终鲜兄弟，门衰祚薄，晚有儿息。
外无期功强近之亲，内无应门五尺之僮，茕茕孑立，形影相吊。
而刘夙婴疾病，常在床蓐，臣侍汤药，未曾废离。
"""

In [23]:
from collections import Counter

special_tokens = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']


def build_vocab(text):
    counter = Counter()
    for word in text:
        counter[word] += 1
    vocab = special_tokens.copy()
    for word, count in counter.items():
        if word not in special_tokens:
            vocab.append(word)
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    return vocab, word2idx, idx2word


vocab, word2idx, idx2word = build_vocab(text)
print(word2idx)
print(idx2word)

{'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4, '\n': 5, '臣': 6, '密': 7, '言': 8, '：': 9, '以': 10, '险': 11, '衅': 12, '，': 13, '夙': 14, '遭': 15, '闵': 16, '凶': 17, '。': 18, '生': 19, '孩': 20, '六': 21, '月': 22, '慈': 23, '父': 24, '见': 25, '背': 26, '；': 27, '行': 28, '年': 29, '四': 30, '岁': 31, '舅': 32, '夺': 33, '母': 34, '志': 35, '祖': 36, '刘': 37, '愍': 38, '孤': 39, '弱': 40, '躬': 41, '亲': 42, '抚': 43, '养': 44, '少': 45, '多': 46, '疾': 47, '病': 48, '九': 49, '不': 50, '零': 51, '丁': 52, '苦': 53, '至': 54, '于': 55, '成': 56, '立': 57, '既': 58, '无': 59, '伯': 60, '叔': 61, '终': 62, '鲜': 63, '兄': 64, '弟': 65, '门': 66, '衰': 67, '祚': 68, '薄': 69, '晚': 70, '有': 71, '儿': 72, '息': 73, '外': 74, '期': 75, '功': 76, '强': 77, '近': 78, '之': 79, '内': 80, '应': 81, '五': 82, '尺': 83, '僮': 84, '茕': 85, '孑': 86, '形': 87, '影': 88, '相': 89, '吊': 90, '而': 91, '婴': 92, '常': 93, '在': 94, '床': 95, '蓐': 96, '侍': 97, '汤': 98, '药': 99, '未': 100, '曾': 101, '废': 102, '离': 103}
{0: '[PAD]', 1: '[UNK]', 2: '[CLS]', 3: '[SEP]'

In [25]:
import random
from torch.utils.data import Dataset, DataLoader
import torch


class BertDataset(Dataset):
    def __init__(self, data, max_seq_len=20):
        self.data = [ch for ch in data if ch not in ('\n', '，', '？', '。', '：', '；')]
        self.max_seq_len = max_seq_len
        self.sentences = [self.data[i:i + 4] for i in range(0, len(self.data), 4)]
        print(len(self.sentences))
        print(self.sentences)

    def __len__(self):
        return len(self.sentences) // 2

    def __getitem__(self, index):
        is_next = random.random() > 0.5

        a_idx = index * 2
        sentence_a = list(self.sentences[a_idx])

        if is_next and a_idx + 1 < len(self.sentences):
            sentence_b = list(self.sentences[a_idx + 1])
        else:
            b_idx = random.randint(0, len(self.sentences) - 1)
            while abs(b_idx - a_idx) <= 1:
                b_idx = random.randint(0, len(self.sentences) - 1)
            sentence_b = list(self.sentences[b_idx])
            is_next = False

        token = ['[CLS]'] + sentence_a + ['[SEP]'] + sentence_b + ['[SEP]']

        if len(token) > self.max_seq_len:
            token = token[:self.max_seq_len - 1] + ['[SEP]']
        elif len(token) < self.max_seq_len:
            token += ['[PAD]'] * (self.max_seq_len - len(token))

        input_ids = [word2idx.get(ch, word2idx['[UNK]']) for ch in token]

        seq_pos = [i for i, token in enumerate(token) if token == '[SEP]']
        segment_ids = [0] * self.max_seq_len
        if len(seq_pos) >= 2:
            for i in range(seq_pos[0] + 1, seq_pos[1] + 1):
                segment_ids[i] = 1

        atention_mask = [1 if token != '[PAD]' else 0 for token in token]
        mlm_labels = [-100] * self.max_seq_len

        maskable_pos = []
        for i, token in enumerate(token):
            if token not in ['[CLS]', '[SEP]', '[PAD]']:
                maskable_pos.append(i)

        num_mask = max(1, int(len(maskable_pos) * 0.15))
        mask_pos = random.sample(maskable_pos, min(num_mask, len(maskable_pos)))
        for i in mask_pos:
            mlm_labels[i] = input_ids[i]

            rand = random.random()
            if rand < 0.8:
                input_ids[i] = word2idx['[MASK]']
            elif rand < 0.9:
                available_ids = [i for i in range(len(vocab)) if
                                 i not in [word2idx['[CLS]'], word2idx['[SEP]'], word2idx['[PAD]'], word2idx['[MASK]']]]
                if available_ids:
                    input_ids[i] = random.choice(available_ids)

        return {
            'input_ids': torch.LongTensor(input_ids),
            'segment_ids': torch.LongTensor(segment_ids),
            'attention_mask': torch.LongTensor(atention_mask),
            'mlm_labels': torch.LongTensor(mlm_labels),
            'nsp_label': torch.LongTensor([int(is_next)]),
        }


dataset = BertDataset(text)
dataLoader = DataLoader(dataset, batch_size=2, shuffle=True)
for batch in dataLoader:
    print(batch)
    break

29
[['臣', '密', '言', '臣'], ['以', '险', '衅', '夙'], ['遭', '闵', '凶', '生'], ['孩', '六', '月', '慈'], ['父', '见', '背', '行'], ['年', '四', '岁', '舅'], ['夺', '母', '志', '祖'], ['母', '刘', '愍', '臣'], ['孤', '弱', '躬', '亲'], ['抚', '养', '臣', '少'], ['多', '疾', '病', '九'], ['岁', '不', '行', '零'], ['丁', '孤', '苦', '至'], ['于', '成', '立', '既'], ['无', '伯', '叔', '终'], ['鲜', '兄', '弟', '门'], ['衰', '祚', '薄', '晚'], ['有', '儿', '息', '外'], ['无', '期', '功', '强'], ['近', '之', '亲', '内'], ['无', '应', '门', '五'], ['尺', '之', '僮', '茕'], ['茕', '孑', '立', '形'], ['影', '相', '吊', '而'], ['刘', '夙', '婴', '疾'], ['病', '常', '在', '床'], ['蓐', '臣', '侍', '汤'], ['药', '未', '曾', '废'], ['离']]
{'input_ids': tensor([[ 2, 46, 47,  4, 49,  3, 67, 68, 69, 70,  3,  0,  0,  0,  0,  0,  0,  0,
          0,  0],
        [ 2, 15, 16, 17, 19,  3,  4, 89, 90, 91,  3,  0,  0,  0,  0,  0,  0,  0,
          0,  0]]), 'segment_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'a

In [26]:
model = BertModel(len(vocab), MAX_SEQ_LENGTH)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

mlm_criterion = nn.CrossEntropyLoss(ignore_index=-100)
nsp_criterion = nn.CrossEntropyLoss()

epoch = 250
for epoch in range(epoch):
    for batch in dataLoader:
        input_ids = batch['input_ids']
        segment_ids = batch['segment_ids']
        attention_mask = batch['attention_mask']
        mlm_labels = batch['mlm_labels']
        nsp_label = batch['nsp_label'].squeeze(1)

        mlm_output, nsp_output = model(input_ids, segment_ids, attention_mask)

        mlm_loss = mlm_criterion(mlm_output.view(-1, mlm_output.size(-1)), mlm_labels.view(-1))
        nsp_loss = nsp_criterion(nsp_output, nsp_label)

        loss = mlm_loss + nsp_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch {epoch}, Loss: {loss.item()}')


Epoch 0, Loss: 6.6515936851501465
Epoch 1, Loss: 5.0840959548950195
Epoch 2, Loss: 4.57070255279541
Epoch 3, Loss: 4.926476001739502
Epoch 4, Loss: 5.696877479553223
Epoch 5, Loss: 4.794158935546875
Epoch 6, Loss: 5.023797512054443
Epoch 7, Loss: 4.047366142272949
Epoch 8, Loss: 5.4910054206848145
Epoch 9, Loss: 5.196789741516113
Epoch 10, Loss: 5.471772193908691
Epoch 11, Loss: 5.765669822692871
Epoch 12, Loss: 4.519994735717773
Epoch 13, Loss: 4.819960594177246
Epoch 14, Loss: 4.826550006866455
Epoch 15, Loss: 3.9310431480407715
Epoch 16, Loss: 5.3567633628845215
Epoch 17, Loss: 4.346385955810547
Epoch 18, Loss: 4.2409892082214355
Epoch 19, Loss: 5.150585651397705
Epoch 20, Loss: 4.52568244934082
Epoch 21, Loss: 4.670572280883789
Epoch 22, Loss: 4.2741289138793945
Epoch 23, Loss: 5.583310127258301
Epoch 24, Loss: 4.424507141113281
Epoch 25, Loss: 4.9732255935668945
Epoch 26, Loss: 4.524213790893555
Epoch 27, Loss: 5.040576934814453
Epoch 28, Loss: 4.602090358734131
Epoch 29, Loss: 5.

In [29]:
model.eval()


def generate(tokens):
    input_ids = [word2idx[token] for token in tokens]
    input_ids += [word2idx['[PAD]']] * (MAX_SEQ_LENGTH - len(tokens))
    segment_ids = [0] * 6 + [1] * 5 + [0] * (MAX_SEQ_LENGTH - 11)
    attention_mask = [1] * 6 + [1] * 5 + [0] * (MAX_SEQ_LENGTH - 11)

    inputs_tensor = torch.LongTensor([input_ids])
    segment_tensor = torch.LongTensor([segment_ids])
    mask_tensor = torch.LongTensor([attention_mask])

    mlm_output, nsp_output = model(inputs_tensor, segment_tensor, mask_tensor)
    print(f"输入序列：{tokens}")

    print("MLM输出：")
    for i, token in enumerate(tokens):
        if token == '[MASK]':
            predicted_id = mlm_output[0][i].argmax().item()
            pred_word = idx2word.get(predicted_id, '[UNK]')
            print(f"位置 {i}的预测结果为{pred_word}")

    nsp_probs = torch.softmax(nsp_output[0], dim=0)
    nsp_prediction = nsp_output[0].argmax().item()

    result = "是" if nsp_prediction == 1 else "不是"
    confidence = nsp_probs[nsp_prediction].item()

    print(f"NSP输出：{result}，概率为{confidence:.4f}")


text = ['[CLS]', '臣', '[MASK]', '言', '臣', '[SEP]', '[MASK]', '险', '衅', '夙', '[SEP]']
generate(text)


输入序列：['[CLS]', '臣', '[MASK]', '言', '臣', '[SEP]', '[MASK]', '险', '衅', '夙', '[SEP]']
MLM输出：
位置 2的预测结果为密
位置 6的预测结果为以
NSP输出：是，概率为0.5646


In [32]:
from modelscope import AutoModel

bert_model=AutoModel.from_pretrained('google-bert/bert-base-uncased')

print(bert_model)


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False